<a href="https://colab.research.google.com/github/Amshu0-0/CA/blob/main/NLP/notebooks/colab_full_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full experiment on a GPU (Google Colab)

This notebook runs the whole project on the **full BGL dataset**:

1. download and preprocess BGL (Drain templates, chronological 60/20/20 split, 5-minute windows)
2. **Approach A** - train LogBERT and tune its detection settings on the validation period
3. **Approach B** - induce rules with an LLM, then run the zero-shot, few-shot, rule-based and keyword conditions
4. compare all methods on the same test windows, run the error analysis, and launch the demo

**Runtime:** choose *Runtime -> Change runtime type -> T4 GPU*. Expect roughly 1-2 hours in total;
the LLM steps dominate. Every LLM answer is cached in `outputs/llm_cache`, so an interrupted run
resumes without paying twice - copy `outputs/` to Google Drive at the end (last cell).

**Before running:** Do not select Run all without reviewing the cells. Training and model downloads use GPU resources, and the OpenAI rule-induction option may incur API charges. Choose only one rule-induction option.


In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 0. Get the code

In [2]:
BRANCH = "main"
!git clone -q -b {BRANCH} https://github.com/Amshu0-0/CA.git
%cd CA/NLP
!pip install -q -e ".[llm,demo]" bitsandbytes

/content/CA/NLP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for logad (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.12.0 requires cachetools>=5.5, but you have cachetools 4.2.1 which is incompatible.


## 1. Data
Downloads BGL (4,747,963 messages) from Loghub's Zenodo record and preprocesses it.

In [ ]:
!python scripts/download_data.py
!python scripts/preprocess.py

07:03:34 | INFO    | logad.data.download | Downloading https://zenodo.org/records/8196385/files/BGL.zip?download=1
download: 100% 57.5M/57.5M [00:16<00:00, 3.42MB/s]


In [ ]:
import json

stats = json.load(open("data/processed/bgl/stats.json"))
print(json.dumps(stats["splits"], indent=2))
print("template purity:", stats["template_purity"])

## 2. Approach A - LogBERT
Training uses only normal windows from the training period; early stopping watches a held-out tail of
those windows. Detection settings `g` and `r` are then tuned on the labeled validation period, for the
original random masking and for our full-coverage extension.

In [ ]:
!python scripts/train_logbert.py
!python scripts/evaluate_logbert.py
!python scripts/evaluate_logbert.py --set logbert.detection.full_coverage=true

## 3. Approach B - rule-guided LLM

### 3a. Rule induction
Rules are induced with **Qwen2.5-7B-Instruct** (4-bit) on this GPU - **no API key needed**. LogRules used
GPT-4o-mini; the proposal allows "a comparable model" when it is unavailable. If you do have an OpenAI
key, store it as a Colab secret named `OPENAI_API_KEY` (key icon in the left sidebar) and set
`USE_GPT4O_MINI = True`.

In [ ]:
USE_GPT4O_MINI = False  # True needs the Colab secret OPENAI_API_KEY

if USE_GPT4O_MINI:
    import os

    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    GENERATOR = "--set llm.generator.model=gpt-4o-mini --set llm.generator.base_url=null --set llm.generator.api_key_env=OPENAI_API_KEY"
else:
    GENERATOR = "--set llm.generator.backend=transformers --set llm.generator.model=Qwen/Qwen2.5-7B-Instruct --set llm.generator.load_in_4bit=true"
!python scripts/induce_rules.py {GENERATOR}

In [ ]:
print(open("outputs/bgl/rules/rules.md").read())

### 3b. The four conditions
The classifier is **Qwen2.5-7B-Instruct**, loaded in 4 bit with `transformers` so it fits on a T4.
Each unique template in the validation and test periods is classified once per condition.

In [ ]:
QWEN_CLS = "--set llm.classifier.backend=transformers --set llm.classifier.model=Qwen/Qwen2.5-7B-Instruct --set llm.classifier.load_in_4bit=true"
!python scripts/run_rule_guided.py {QWEN_CLS}

## 4. Comparison and error analysis

In [ ]:
!python scripts/compare.py
!python scripts/analyze.py

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("outputs/bgl/results/comparison.md").read()))
display(Markdown(open("outputs/bgl/analysis/analysis.md").read()))

## 5. Keep the results
Copies `outputs/` (checkpoint, rules, predictions, metrics, LLM cache) to Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/CA-NLP
!cp -r outputs /content/drive/MyDrive/CA-NLP/

## 6. Live demo
Prints a public `gradio.live` link you can share with the class (valid for 72 hours).

In [ ]:
!python app/demo.py --share {QWEN_CLS}